# Install the `kaipy` Python package.

In [ ]:
!pip install kaipy

# Import required modules.

In [ ]:
# Standard modules
import os

# 3rd party modules
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec

# Project modules
import kaipy.gamera.magsphere as msph
import kaipy.gamera.msphViz as mviz
import kaipy.kaiViz as kv
import kaipy.remix.remix as remix

# Visualize geospace using GAMERA model results.

### Data files from a typical MAGE model run on `derecho` are already uploaded to scratch space in HelioCloud. This is a relatively small set of results (about 19 GB), for demonstration purposes. But MAGE results from larger runs can reach hundreds of GB in size, and so can be impractical do download to a local machine. For computational efficiency, the magnetosphere results (from the GAMERA component of MAGE) are split across multiple files representing different subsections of the LFM (Lyon-Fedder-Mobarry) grid used for the modeling. The `kaipy` software provides a software layer that hides this complexity, allowing you to treat the entire set of results as a single logical data source. The `kaipy` package is built on top of `numpy`, and so `kaipy`-derived values can be used directly in any other `numpy`-based code (such as `matplotlib`) with no conversion.

## Open a connection to the data.

In [ ]:
# Specify the path to the data directory.
data_directory = os.path.join(os.environ["HOME"], "scratch_space", "gem2026", "cgs_data")

# Specify the tag that identifies the run.
run_name = "msphere"

# Now open a connection to the magnetosphere data using the `kaipy.gamera.magsphere.GamsphPipe` object.
gsph = msph.GamsphPipe(data_directory, run_name)

## Plot the number density in the magnetosphere in a meridional slice.

### The `msphViz` module of the `kaipy` package has numerous routines for visualizing the magnetosphere.  Here will demostrate how to use `plotXZ()` to plot a meridional cut of the number density in the magnetosphere.  For this routine and others in `kaipy` you need to provide the instance of the `GamsphPipe` object which connects to the directory of data, the step number you want to display, the extent of the domain, and two `Axes` objects (one for the plot and one for the colorbar).

In [ ]:
# Plot the number density (D is the default if otherwise unspecified, we specify it here for clarity).
variable_to_plot = "D"

# Specify the index of the time step to plot.
step_to_plot = 109

# Specify the region to plot. Coordinates are in the GSM (Geocentric Solar Magnetic) frame, units of Earth radius.
X_min, X_max = -100.0, 20.0
Z_min, Z_max = -60.0, 60.0
plot_boundaries = [X_min, X_max, Z_min, Z_max]

# Create the matplotlib Figure object. Specify figure size (width, height) in inches.
fig = plt.figure(figsize=(8, 8))

# Create a layout and Axes for the plot body (on top) and the color bar (on bottom).
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[20, 1], hspace=0.2)
plot_axes = fig.add_subplot(gs[0, 0])
colorbar_axes = fig.add_subplot(gs[1, 0])

# Plot the number density from the last time step in the vertical slice. Limit values to the range [0, 50].
# The plot function returns an np.ndarray of the plotted data.
data = mviz.plotXZ(gsph=gsph, var=variable_to_plot, nStp=step_to_plot, xzBds=plot_boundaries, Ax=plot_axes, AxCB=colorbar_axes, vMin=0, vMax=50)

## Now plot an equatorial slice at step 50 showing the solar wind velocity in the equatorial plane ($V_X$).

In [ ]:
# Plot the x-component of the solar wind velocity.
variable_to_plot = "Vx"

# Specify the index of the time step to plot.
step_to_plot = 50
    
# Specify the region to plot. Coordinates are in the GSM (Geocentric Solar Magnetic) frame, units of Earth radius.
X_min, X_max = -100.0, 20.0
Y_min, Y_max = -60.0, 60.0
plot_boundaries = [X_min, X_max, Z_min, Z_max]

# Create the matplotlib Figure object. Specify figure size (width, height) in inches.
fig = plt.figure(figsize=(8, 8))

# Create a layout for the plot body and the color bar.
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[20, 1], hspace=0.2)
plot_axes = fig.add_subplot(gs[0, 0])
colorbar_axes = fig.add_subplot(gs[1, 0])

# Plot the solar wind Vx in the equatorial slice. Customize the plot with a different color map and midpoint normalization.
data = mviz.plotXY(gsph=gsph, var=variable_to_plot, nStp=step_to_plot, xyBds=plot_boundaries, Ax=plot_axes, AxCB=colorbar_axes, midp=True, cmap="RdBu_r")

# Visualize the ionosphere with REMIX model results.

## Open a connection to the data

### Ionosphere results are generated by the REMIX component of the MAGE model. These results are not split - they are provided in a single (large) file, but the `kaipy` software simplifies access by hiding the HDF5 structure of the file. Importing the ionospheric data from REMIX follows a slightly different pattern from the import of the magnetospheric data from GAMERA, with the added requirement of specifying which hemisphere, e.g. `NORTH` or `SOUTH`, that you want to use.

In [ ]:
mixFiles = os.path.join(data_directory, f"{run_name}.mix.h5")
step_to_plot = 109
ion = remix.remix(mixFiles, step_to_plot)
ion.init_vars("NORTH")

## Plot the ionospheric current density in the northern hemisphere.

### The `remix` object includes extensive plottings routine that can plot numerous variables with reasonable default options. `remix` objects can also calculate dervied quantities, such as magnetic perturbations and electric fields.  Unlike the magnetosphere plotting routines it has the option to take a `gridspec` object instead of a pair of `Axes` objects.

In [ ]:
ion.plot("current")

## Plot the north polar particle energy.

### The `remix` routines use a different method for specifying the data limits.  Instead of having limmits passed as parameters from the calling function, each variable has a `dict` associated with it to specify the minimum and maximum values to plot.

In [ ]:
ion.variables["energy"]["min"] = 0.0   # keV
ion.variables["energy"]["max"] = 15.0  # keV
ion.plot("energy")

## Plot the northern cross-polar cap potential.

In [ ]:
ion.plot("potential")

# Make a combination plot of pressure and field-aligned current.

### Magnetosphere and ionosphere plots can easily be combined into a single plot. Note the utilization of `gridspec` and `add_subplot()` to control the location of the main plot and related colorbars.  The color bar object is not returned by the `remix` plotting routines so it must be recreated with `kv.genCB()`.  

In [ ]:
# Specify the index of the time step to plot.
step_to_plot = 109

# Specify the region to plot. Coordinates are in the GSM (Geocentric Solar Magnetic) frame, units of Earth radius.
X_min, X_max = -100.0, 20.0
Z_min, Z_max = -60.0, 60.0
plot_boundaries = [X_min, X_max, Z_min, Z_max]

# Create the matplotlib Figure object. Specify figure size (width, height) in inches.
fig = plt.figure(figsize=(8, 8))

# Create the layout inside the figure.
gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[20, 1])
plot_axes = fig.add_subplot(gs[0, :])
colorbar_axes = fig.add_subplot(gs[1, 0])
AxC2 = fig.add_subplot(gs[1, 1])

# Create the magnetospheric pressure plot.
data = mviz.plotXZ(gsph=gsph, var="P", nStp=step_to_plot, xzBds=plot_boundaries, Ax=plot_axes, AxCB=colorbar_axes)

# Add the north-polar FAC plot as an inset.
plot_axisInset = gs[0, :].subgridspec(20, 20)
wXY = 6
dX = 2
dY = 1
AxIon = ion.plot("current", gs=plot_axisInset[dY:dY+wXY, dX:dX + wXY], doInset=True)
cbM = kv.genCB(AxC2, kv.genNorm(remix.facMax), "FAC", cM=remix.facCM, Ntk=4)
plt.show()